# NOAI Day 3 — Homework (scikit-learn)

**After Day 3 class.** Six problems on the 4-topic core: **Preprocessing · Pipeline · Regression · Pitfalls**.

Vocab: `train_test_split`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `ColumnTransformer`, `Pipeline`, `LogisticRegression`, `LinearRegression`, `r2_score`, `mean_absolute_error`.

Rules: **fit on train, transform everywhere, Pipeline always.** Always `random_state=42`. Built-in datasets only — runs offline.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/noai_day3_homework.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
print("ready")

---
## Problem 1 — Leak-proof scaling, by hand (no `Pipeline`)

**Scenario.** Before you're allowed to use `Pipeline`, prove you understand
*why* it exists. Scale features manually and show the scaler never sees test
data.

Dataset: built-in **diabetes** features. Split is given (`random_state=42`).

Do this:
1. Fit a `StandardScaler` **on `X_tr` only** → `sc`.
2. `X_tr_s` = scaled train features; `X_te_s` = scaled test features
   (transform both with the **same** `sc`).
3. `learned_mean0` = `sc.mean_[0]`.

Check that `learned_mean0` equals the raw **training** column-0 mean (it learned
from train only) and that scaled train data is centred (`X_tr_s.mean() ≈ 0`).

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
# TODO
sc = ...
X_tr_s = ...
X_te_s = ...
learned_mean0 = ...
print(round(learned_mean0, 6), round(X_tr[:, 0].mean(), 6))

In [ ]:
assert abs(learned_mean0 - X_tr[:, 0].mean()) < 1e-9
assert abs(X_tr_s.mean()) < 1e-9
print("Q1 ok")

<details><summary>Hint</summary>

```python
sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr)
X_te_s = sc.transform(X_te)
learned_mean0 = sc.mean_[0]
```

</details>

---
## Problem 2 — Impute + one-hot a messy frame

**Scenario.** A colleague hands you a small customer table with one missing age
and one missing city. Clean it into a fully numeric array, ready for a model.

```
age:  [22, NaN, 35, 40, 28]
city: ["A", "B", "A", None, "B"]
```

Do this:
1. `num` pipe (for `age`): `SimpleImputer(median)` → `StandardScaler`.
2. `cat` pipe (for `city`): `SimpleImputer(most_frequent)` →
   `OneHotEncoder(handle_unknown="ignore")`.
3. Combine with a `ColumnTransformer`, then `fit_transform` the frame.
4. `Xt` = the dense output array; `n_out` = its column count.

Expected: 5 rows, `n_out == 3` (1 numeric + 2 city dummies).

In [ ]:
df = pd.DataFrame({"age":  [22., np.nan, 35., 40., 28.],
                   "city": ["A", "B", "A", None, "B"]})
# TODO
num = ...
cat = ...
pre = ...
Xt  = ...
n_out = ...
print(Xt.shape, n_out)

In [ ]:
assert Xt.shape[0] == 5
assert n_out == 3          # 1 numeric + 2 city dummies
print("Q2 ok")

<details><summary>Hint</summary>

```python
num = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())])
cat = Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                ("o", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("n", num, ["age"]), ("c", cat, ["city"])])
Xt = pre.fit_transform(df)
n_out = Xt.shape[1]
```

</details>

---
## Problem 3 — Full classification pipeline (Wine)

**Scenario.** You need a clean, end-to-end, leak-proof classifier you could
hand to production — no manual scaling, everything inside one `Pipeline`.

Dataset: built-in **wine** (predict the cultivar from chemical measurements).

Do this:
1. `load_wine(return_X_y=True)`, split 80/20 with `stratify=y`,
   `random_state=42`.
2. `Pipeline`: `SimpleImputer(median)` → `StandardScaler` →
   `LogisticRegression(max_iter=5000)`. Fit on train.
3. `acc` = test accuracy.

Target: `acc > 0.95`.

In [ ]:
X, y = load_wine(return_X_y=True)
# TODO
X_tr, X_te, y_tr, y_te = ...
pipe = ...
acc = ...
print("Wine pipeline acc:", round(acc, 3))

In [ ]:
assert acc > 0.95
print("Q3 ok")

<details><summary>Hint</summary>

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([("i", SimpleImputer(strategy="median")),
                 ("s", StandardScaler()),
                 ("m", LogisticRegression(max_iter=5000))]).fit(X_tr, y_tr)
acc = pipe.score(X_te, y_te)
```

</details>

---
## Problem 4 — Regression, reported correctly (R² / MAE / RMSE)

**Scenario.** A teammate reported "95% accuracy" on a regression model — that's
meaningless, accuracy is for classification. Show the *correct* regression
metrics instead.

Dataset: built-in **diabetes** (predict a number). Split `random_state=42`.

Do this:
1. `Pipeline(StandardScaler → LinearRegression)`, fit on train.
2. Predict test into `yp`.
3. `r2`, `mae`, `rmse` (RMSE = sqrt of mean squared error).
4. `score` = `pipe.score(X_te, y_te)`.

Show that `score == r2` (a regressor's `.score()` *is* R²). Need `r2 > 0.4`.

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
# TODO
pipe = ...
yp = ...
r2 = ...
mae = ...
rmse = ...
score = ...
print(round(r2, 3), round(mae, 1), round(rmse, 1))

In [ ]:
assert abs(score - r2) < 1e-9
assert r2 > 0.4
print("Q4 ok")

<details><summary>Hint</summary>

```python
pipe = Pipeline([("s", StandardScaler()), ("m", LinearRegression())]).fit(X_tr, y_tr)
yp = pipe.predict(X_te)
r2 = r2_score(y_te, yp)
mae = mean_absolute_error(y_te, yp)
rmse = mean_squared_error(y_te, yp) ** 0.5
score = pipe.score(X_te, y_te)
```

</details>

---
## Problem 5 — Fix the leaky code

**Scenario.** You're reviewing a teammate's pull request. Their code scales the
data **before** the split, then **refits** a scaler on the test set — both leak
test information into training. Reject it and rewrite it correctly.

```python
# BUGGY — do NOT submit this
sc = StandardScaler()
Xs = sc.fit_transform(X)                       # sees ALL data (leak #1)
X_tr, X_te, y_tr, y_te = train_test_split(Xs, y, random_state=42)
X_te = StandardScaler().fit_transform(X_te)    # refit on test (leak #2)
```

Rewrite it leak-proof on the **wine** dataset using a single `Pipeline` (so
neither leak is even possible). Split `stratify=y`, `random_state=42`, set
`acc`. Target: `acc > 0.9`.

In [ ]:
X, y = load_wine(return_X_y=True)
# TODO  — leak-proof, Pipeline only
X_tr, X_te, y_tr, y_te = ...
pipe = ...
acc = ...
print("fixed acc:", round(acc, 3))

In [ ]:
assert acc > 0.9
print("Q5 ok")

<details><summary>Hint</summary>

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([("s", StandardScaler()),
                 ("m", LogisticRegression(max_iter=5000))]).fit(X_tr, y_tr)
acc = pipe.score(X_te, y_te)
```

</details>

---
## Problem 6 — Mixed-type, end-to-end (the production template)

**Scenario.** A Titanic-style table: passenger `age`/`fare` (some missing) and
`sex`/`port` (some missing). Build the full production template that treats
numeric and categorical columns differently — all in one `Pipeline`.

```
age:  [22, 38, NaN, 35, 28, 40, 50, 8]
fare: [7.2, 71.3, 8.0, NaN, 13, 27, 9.5, 21]
sex:  ["m","f","f","f","m","m","m",None]
port: ["S","C","S",None,"S","Q","S","C"]
```

Do this:
1. `num` pipe (`age`,`fare`): impute(median) → scale.
2. `cat` pipe (`sex`,`port`): impute(most_frequent) → OneHot.
3. `ColumnTransformer` over `num_cols`/`cat_cols`, wrapped in a full
   `Pipeline` → `LogisticRegression(max_iter=1000)`.
4. Split `stratify=y`, `random_state=42`, fit, set `acc`.

Any `acc` in `[0, 1]` passes — the data is tiny; the graded part is the
correct structure.

In [ ]:
df = pd.DataFrame({
    "age":  [22., 38., np.nan, 35., 28., 40., 50., 8.],
    "fare": [7.2, 71.3, 8.0, np.nan, 13.0, 27.0, 9.5, 21.0],
    "sex":  ["m", "f", "f", "f", "m", "m", "m", None],
    "port": ["S", "C", "S", None, "S", "Q", "S", "C"],
})
y = np.array([0, 1, 1, 1, 0, 0, 0, 1])
num_cols = ["age", "fare"]
cat_cols = ["sex", "port"]
# TODO
num = ...
cat = ...
pre = ...
pipe = ...
X_tr, X_te, y_tr, y_te = ...
acc = ...
print(round(acc, 3))

In [ ]:
assert 0.0 <= acc <= 1.0
print("Q6 ok")

<details><summary>Hint</summary>

```python
num = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())])
cat = Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                ("o", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
pipe = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])
X_tr, X_te, y_tr, y_te = train_test_split(
    df, y, test_size=0.25, random_state=42, stratify=y)
pipe.fit(X_tr, y_tr)
acc = pipe.score(X_te, y_te)
```

</details>